# FlytBase Hackathon — Colab Starter

Run tonight to (1) confirm GPU access, (2) install the core stack once, and (3) warm a
**persistent** model cache on Google Drive so tomorrow's fresh Colab session doesn't have to
re-download multi-GB weights from HuggingFace at 11:00 AM while every other team is also
hitting HF's servers.

Before running: **Runtime -> Change runtime type -> T4 GPU -> Save.**

If you get Kaggle working instead: same cells work, just replace the Drive-mount cell with
`CACHE_DIR = "/kaggle/working/model_cache"` (Kaggle persists `/kaggle/working/` across a
saved-version, no Drive needed).

## 1. GPU check (the one thing the organizers' guide actually requires)

In [ ]:
import os
# Reduces allocator fragmentation - cheap insurance against an OOM found on the Kaggle side
# tonight. Must be set before torch touches CUDA.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1b. Pin torch/torchvision before installing anything else

**Found on the Kaggle side tonight, same risk applies here**: an unconstrained `pip install`
of `accelerate`, `supervision`, or transformers-from-source can silently upgrade `torch` to a
build whose compiled CUDA kernels don't match this container's actual GPU - it loads and runs
fine right up until the first real inference call, which then crashes with
`AcceleratorError: CUDA error: no kernel image is available for execution on the device`.
No warning at install time, unlike the pandas issue - it fails deep inside a forward pass.

Fix: capture whatever torch/torchvision are *already* correctly matched to this GPU right
now, and pin them explicitly in every subsequent install so pip's resolver can never touch
them.

In [ ]:
import torch
TORCH_PIN = torch.__version__
try:
    import torchvision
    TORCHVISION_PIN = torchvision.__version__
except ImportError:
    TORCHVISION_PIN = None

pins = f'"torch=={TORCH_PIN}"'
if TORCHVISION_PIN:
    pins += f' "torchvision=={TORCHVISION_PIN}"'

print("Pinning for all subsequent installs:", pins)

## 2. Persistent storage

Colab's local disk is wiped every session. Mount Drive so downloaded model weights survive
a disconnect — tomorrow's session just re-mounts and finds everything already cached.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = "/content/drive/MyDrive/flytbase_hackathon/model_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Point every HF download at the persistent cache instead of local (ephemeral) disk.
os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_CACHE"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
print("Model cache ->", CACHE_DIR)

## 3. Core dependencies (fast installs)

This is the lean set needed for tonight's smoke test + tomorrow's ingest/enrich/index stages.
Not included yet (add later only if we actually use them, to keep this cell fast):
`lancedb`, `fiftyone`, `gradio`, `faster-whisper`, `open_clip_torch`, `facebook/sam3`.

**Deliberately not touching `pandas`/`pyarrow` or using `-U` here.** Colab ships
pandas/pyarrow versions that `google-colab` and the RAPIDS cuDF stack are pinned to
exactly (`pandas==2.2.3`); a blanket `-U` on an explicitly-listed `pandas` will upgrade
past that and break both. If you ever add a package here that transitively wants a newer
pandas, pin it back afterwards: `!pip install -q "pandas==2.2.3"` then Runtime -> Restart
session (not a full disconnect - that loses the GPU allocation and cached installs).
`{pins}` from the previous cell locks torch/torchvision the same way.

In [ ]:
!pip install -q accelerate bitsandbytes faiss-cpu decord av scenedetect supervision \
    rank_bm25 rapidocr-onnxruntime pillow opencv-python-headless {pins}

## 4. Transformers — built from source

Qwen3-VL's model class isn't in the last stable `transformers` release yet per its model
card, so this installs straight from the GitHub main branch. Takes a couple of minutes
(it's a pip/wheel build, not a C++ compile — just slower than a normal `pip install`).

**Not `--no-deps`** (tried that on the Kaggle side, it broke: transformers-main wants
`tokenizers>=0.23.1,<0.24.0` and without letting pip resolve that, it left an older
`tokenizers` in place, which then fails at import time). Instead, pass the same `{pins}`
used elsewhere — pip resolves everything else (tokenizers, huggingface_hub, safetensors...)
normally to satisfy what transformers actually needs, while torch/torchvision stay locked to
the exact already-working versions no matter what.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers {pins}

## 5. Smoke test 1 — SigLIP2 (retrieval embeddings)

Confirms the embedding model our `index.py`/`search.py` will rely on actually loads and
produces sane image embeddings on this hardware.

Pinned to an explicit device (`cuda:0`) rather than `device_map="auto"`, and explicitly
deleted + `empty_cache()`'d at the end: on Kaggle, leaving this resident caused Qwen3-VL to
OOM even with a second GPU sitting empty. Colab only has one GPU, so freeing this before the
next cell loads Qwen3-VL matters even more here.

In [ ]:
import gc
import torch
from transformers import AutoModel, AutoProcessor
from transformers.image_utils import load_image

SIGLIP_CKPT = "google/siglip2-so400m-patch14-384"
siglip_model = AutoModel.from_pretrained(SIGLIP_CKPT).to("cuda:0").eval()
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_CKPT)

image = load_image(
    "https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000000285.jpg"
)
inputs = siglip_processor(images=[image], return_tensors="pt").to(siglip_model.device)
with torch.no_grad():
    image_output = siglip_model.get_image_features(**inputs)

# This transformers-main build wraps the result in BaseModelOutputWithPooling instead of
# returning a bare tensor - handle both so a future API change doesn't break this again.
image_embedding = getattr(image_output, "pooler_output", image_output)

print("SigLIP2 loaded OK. Embedding shape:", image_embedding.shape)

# Free the GPU before the next model loads - see markdown above for why this matters.
del siglip_model, inputs, image_output, image_embedding
gc.collect()
torch.cuda.empty_cache()

## 6. Smoke test 2 — Qwen3-VL-4B-Instruct (captioning)

Confirms the captioning/VQA model loads and generates on a T4. This is the biggest download
(a few GB) — the whole point of running it tonight is so it's cached on Drive by tomorrow.

Uses `cuda:1` if a second GPU exists, else `cuda:0` (Colab's normal case - relies on the
previous cell's cleanup having freed it).

**4-bit quantized + downscaled input image.** Found on the Kaggle side tonight: full-precision
4B weights (~8 GiB) plus one vision-tower attention call can spike to 7+ GiB in a single
allocation - together that doesn't fit in a T4's ~15 GiB even with nothing else on the GPU.
Two independent fixes, both worth keeping for the real pipeline tomorrow (not just this
test): 4-bit quantization shrinks the weights to ~2 GiB, and capping the input image's
longest side directly caps the vision tower's sequence length - relevant since real
CCTV/drone frames will be arbitrary, often large, resolutions.

Note: model-loading APIs for very recent releases can shift. If `Qwen3VLForConditionalGeneration`
has been renamed by the time you run this, check the "Use this model" snippet on the model's
HuggingFace page directly — that page is always the current source of truth, this notebook
reflects it as of tonight.

In [ ]:
import io
import requests
from PIL import Image
from transformers import (
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
    AutoProcessor as QwenProcessor,
)

QWEN_CKPT = "Qwen/Qwen3-VL-4B-Instruct"
qwen_device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_CKPT, quantization_config=quant_config, device_map=qwen_device, trust_remote_code=True
)
qwen_processor = QwenProcessor.from_pretrained(QWEN_CKPT, trust_remote_code=True)

demo_image_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
demo_image = Image.open(io.BytesIO(requests.get(demo_image_url).content)).convert("RGB")
demo_image.thumbnail((512, 512))  # caps vision-tower sequence length - see markdown above

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": demo_image},
            {"type": "text", "text": "Describe this image in one sentence."},
        ],
    }
]

qwen_inputs = qwen_processor.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
).to(qwen_model.device)

generated_ids = qwen_model.generate(**qwen_inputs, max_new_tokens=64)
output_text = qwen_processor.batch_decode(
    generated_ids[:, qwen_inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print("Qwen3-VL-4B loaded OK. Caption:", output_text[0])

## Done

If both smoke tests printed output above with no errors: GPU works, both core models load
and run, and their weights are now cached under `/content/drive/MyDrive/flytbase_hackathon/model_cache`.

**Tomorrow morning**, in a fresh Colab session, you only need to re-run cells 1, 2, and 3
(GPU check, Drive mount + cache env vars, pip installs) before pointing this at real footage —
model loading will pull from the Drive cache instead of re-downloading.

Not done here on purpose (belongs to Task 2 / Task 4, once real footage exists at 11:00):
the actual `ingest.py` -> `enrich.py` -> `index.py` -> `search.py` -> `answer.py` pipeline,
and `evaluate.py` with real ground-truth queries.